# Publish MM-CAD checkpoints to Hugging Face

Mounts Drive, verifies the three released checkpoints, strips optimizer state,
writes a model card, and uploads everything to `exanos/MMCAD` under `checkpoints/`.

Runs entirely inside Colab, so the multi-GB transfer is Google -> Hugging Face
and never touches your home connection.

**Your HF token never appears in this notebook.** Put it in Colab Secrets
(key icon in the left sidebar) under the name `HF_TOKEN`, with *Notebook access*
enabled. It must be a **write** token.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/MMCAD'
REPO  = 'exanos/MMCAD'      # dataset repo -- checkpoints land under checkpoints/
PREFIX = 'checkpoints'

!pip install -q -U "huggingface_hub[cli]" torch


## 1. The three released checkpoints

These are the ones the paper's numbers come from. Everything else in the Drive
folder (`baseline_trimodal_v4_C`, `_C_ft`, `vocab_*`, `exp*`, `rung1`,
`vol_decomp`) belongs to the post-paper geometric-vocabulary work and is **not**
part of this release.

In [ ]:
import os

CKPTS = {
    'baseline_trimodal_v4.pth': f'{DRIVE}/baseline_trimodal_v4/latest.pth',
    'sketch_encoder_v1.pth':    f'{DRIVE}/sketch_encoder_v1/best.pth',
    'render_encoder_v1.pth':    f'{DRIVE}/render_encoder_v1/best.pth',
}

for name, path in CKPTS.items():
    ok = os.path.exists(path)
    size = f'{os.path.getsize(path)/1e9:.2f} GB' if ok else '--'
    print(f'{"OK " if ok else "MISSING"}  {name:<28} {size:>9}  {path}')


## 2. Inspect before uploading

Reads only each archive's pickle header, so nothing large is loaded. Confirm the
towers and metrics look right before going further.

In [ ]:
import zipfile, pickletools, io, re

def header(path):
    with zipfile.ZipFile(path) as z:
        blob = z.read(next(n for n in z.namelist() if n.endswith('data.pkl')))
    out = io.StringIO()
    try: pickletools.dis(blob, out)
    except Exception: pass
    toks = []
    for line in out.getvalue().split('\n'):
        m = re.search(r"(?:SHORT_BINUNICODE|BINUNICODE)\s+'([^']*)'", line)
        if m: toks.append(('S', m.group(1))); continue
        m = re.search(r'BINFLOAT\s+([-\d.eE+]+)', line)
        if m: toks.append(('F', float(m.group(1)))); continue
        m = re.search(r'BININT\d?\s+(-?\d+)', line)
        if m: toks.append(('I', int(m.group(1))))
    roots = sorted({t.split('.')[0] for k, t in toks if k == 'S' and '.' in t})
    metrics = {}
    for i in range(len(toks) - 1):
        (k1, key), (k2, val) = toks[i], toks[i+1]
        if k1 == 'S' and k2 in ('F', 'I') and re.search(r'R@|recall|mAP|epoch', str(key), re.I):
            metrics.setdefault(key, val)
    return roots, metrics

for name, path in CKPTS.items():
    if not os.path.exists(path):
        continue
    roots, metrics = header(path)
    print('=' * 70)
    print(name)
    print('  module roots:', roots[:6])
    print('  epoch:', metrics.get('epoch'))
    top = sorted(((k, v) for k, v in metrics.items()
                  if k != 'epoch' and re.search(r'R@|recall|mAP', k, re.I)),
                 key=lambda kv: -kv[1])[:8]
    for k, v in top:
        print(f'      {k:<34} {v:.2f}')


## 3. Strip optimizer state

Training checkpoints carry optimizer moments that are useless for inference and
often account for over half the file. This keeps `model_state_dict` plus the
small metadata keys, and writes float32 weights.

In [ ]:
import torch, shutil

STAGE = '/content/mmcad_release'
os.makedirs(STAGE, exist_ok=True)

KEEP = {'model_state_dict', 'epoch', 'config', 'log_tau', 'matryoshka_dims'}

for name, path in CKPTS.items():
    if not os.path.exists(path):
        print('skip (missing):', name); continue
    ck = torch.load(path, map_location='cpu', weights_only=False)
    if isinstance(ck, dict) and 'model_state_dict' in ck:
        slim = {k: v for k, v in ck.items() if k in KEEP}
        # keep every stored metric too -- they document the release
        slim['metrics'] = {k: v for k, v in ck.items()
                           if isinstance(v, (int, float)) and k not in KEEP}
    else:
        slim = {'model_state_dict': ck}
    out = f'{STAGE}/{name}'
    torch.save(slim, out)
    print(f'{name}:  {os.path.getsize(path)/1e9:.2f} GB -> {os.path.getsize(out)/1e9:.2f} GB')
    del ck, slim


## 4. Model card

Written next to the weights so the checkpoints are self-documenting.

In [ ]:
CARD = '''# MM-CAD retrieval checkpoints

Reference checkpoints for *MM-CAD: A Multi-Modal CAD Dataset and Benchmark for
Cross-Modal Geometric Learning* (SGP 2026 / Computer Graphics Forum).

- Paper: https://doi.org/10.1111/cgf.70523
- Project page: https://exanos.github.io/MMCAD
- Code: https://github.com/exanos/MMCAD

## Files

| File | Towers | Role |
|---|---|---|
| `baseline_trimodal_v4.pth` | text (EmbeddingGemma-300M), B-Rep (BRepFormer), point cloud (DGCNN) | Stage 1, trained jointly with symmetric InfoNCE at all four Matryoshka scales |
| `sketch_encoder_v1.pth` | sketch (ViT-Base) | Stage 2, aligned to frozen B-Rep anchors |
| `render_encoder_v1.pth` | photorealistic image (SigLIP-Base) | Stage 2, aligned to frozen B-Rep anchors |

All encoders share one d=768 space, Matryoshka-nested over {128, 256, 512, 768}.
Optimizer state has been stripped; `metrics` in each file records the validation
scores at the released epoch.

## Retrieval, MM-CAD:B validation (B-Rep gallery, N=19,263, d=768)

| Text | Sketch | Image | R@1 | R@5 | R@10 | R@25 |
|---|---|---|---|---|---|---|
| Y | - | - | 15.42 | 34.86 | 44.61 | 57.93 |
| - | Y | - | 32.69 | 57.86 | 67.31 | 78.59 |
| - | - | Y | 33.20 | 55.24 | 64.20 | 74.81 |
| Y | Y | - | 39.97 | 66.07 | 74.46 | 83.69 |
| Y | - | Y | 38.10 | 61.94 | 70.63 | 80.30 |
| - | Y | Y | 41.81 | 66.31 | 74.68 | 83.79 |
| **Y** | **Y** | **Y** | **45.91** | **70.29** | **77.86** | **85.91** |

Query vectors are summed and renormalized; there is no learned fusion head.
Truncating to d=128 moves trimodal R@1 only from 45.91 to 45.50, so a d=128
FAISS index serves interactive queries at negligible cost.

Known limitation: retrieval matches silhouette, not feature. "Herringbone gear,
ten lightening holes" returns a water-bottle base at rank 1 (correct gear at
#77). This gap is released as a benchmark task.

## Usage

```bash
python inference.py --query "servo mount with four bolt holes" --dim 128
```

See `inference.py` in the GitHub repository.

## License

CC BY-NC 4.0 for project-created artifacts. Underlying geometry remains subject
to each source dataset's terms, including ABC/Onshape terms. See `LICENSES.md`.

## Citation

```bibtex
@article{bharathi2026mmcad,
  title     = {MM-CAD: A Multi-Modal CAD Dataset and Benchmark for Cross-Modal Geometric Learning},
  author    = {Bharathi, Anush and Ananthakrishnan, A and Muthuganapathy, Ramanathan},
  journal   = {Computer Graphics Forum},
  year      = {2026},
  publisher = {Wiley},
  volume    = {45},
  number    = {5},
  doi       = {10.1111/cgf.70523},
  note      = {Proc. SGP 2026}
}
```
'''

open(f'{STAGE}/README.md', 'w').write(CARD)
print(sorted(os.listdir(STAGE)))


## 5. Upload

The token is read from Colab Secrets and passed straight to the client -- it is
never printed and never written to disk.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get('HF_TOKEN'))

api.upload_folder(
    folder_path=STAGE,
    path_in_repo=PREFIX,
    repo_id=REPO,
    repo_type='dataset',
    commit_message='Add released retrieval checkpoints (joint tri-modal, sketch, render)',
)

print(f'https://huggingface.co/datasets/{REPO}/tree/main/{PREFIX}')


## 6. Optional — precomputed gallery embeddings

`inference.py` can load an `.npz` of gallery embeddings so users do not have to
encode 192K B-Reps themselves. If you already cached them during evaluation,
drop them in and re-run the upload cell.

In [ ]:
import numpy as np

# Example: embeddings dict {uid: np.ndarray(768)} cached during evaluation
# np.savez_compressed(f'{STAGE}/gallery_brep.npz',
#                     embeddings=np.stack(list(brep_embs.values())).astype('float32'),
#                     uids=np.array(list(brep_embs.keys())))
# api.upload_file(path_or_fileobj=f'{STAGE}/gallery_brep.npz',
#                 path_in_repo=f'{PREFIX}/gallery_brep.npz',
#                 repo_id=REPO, repo_type='dataset')
print('fill in from your evaluation cache')
